In [1]:
print('In the name of Allah')

In the name of Allah


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [6]:
df = pd.read_csv('titanic_data_updated.csv')

df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
650,651,no,third,"Mitkoff, Mr. Mito",male,NaN,0,0,349221,7.8958,NaN,S
364,365,no,third,"O'Brien, Mr. Thomas",male,NaN,1,0,370365,15.5000,NaN,Q
447,448,yes,first,"Seward, Mr. Frederic Kimber",male,34.0,0,0,113794,26.5500,NaN,S
262,263,no,first,"Taussig, Mr. Emil",male,52.0,1,1,110413,79.6500,E67,S
237,238,yes,second,"Collyer, Miss. Marjorie ""Lottie""",female,8.0,0,2,C.A. 31921,26.2500,NaN,S


In [7]:
df['Cabin'].info()


<class 'pandas.core.series.Series'>
RangeIndex: 891 entries, 0 to 890
Series name: Cabin
Non-Null Count  Dtype 
--------------  ----- 
204 non-null    object
dtypes: object(1)
memory usage: 7.1+ KB


In [8]:
df['Family_size'] = df['SibSp'] + df['Parch'] + 1

df['Cabin'] = df['Cabin'].fillna('Missing')

df['Deck'] = df['Cabin'].astype(str).str[0]

df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_size,Deck
530,531,yes,second,"Quick, Miss. Phyllis May",female,2.0,1,1,26360,26.0000,Missing,S,3,M
341,342,yes,first,"Fortune, Miss. Alice Elizabeth",female,24.0,3,2,19950,263.0000,C23 C25 C27,S,6,C
275,276,yes,first,"Andrews, Miss. Kornelia Theodosia",female,63.0,1,0,13502,77.9583,D7,S,2,D
708,709,yes,first,"Cleaver, Miss. Alice",female,22.0,0,0,113781,151.5500,Missing,S,1,M
441,442,no,third,"Hampe, Mr. Leon",male,20.0,0,0,345769,9.5000,Missing,S,1,M


In [9]:
df['Deck'].value_counts()

,count
Deck,
M,687
C,59
B,47
D,33
E,32
A,15
F,13
G,4
T,1


In [10]:
X = df.drop('Survived', axis = 1)
y = df['Survived']

In [11]:
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size = 0.2, random_state = 42, stratify = y )

In [12]:
X_train

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_size,Deck
692,693,third,"Lam, Mr. Ali",male,NaN,0,0,1601,56.4958,Missing,S,1,M
481,482,second,"Frost, Mr. Anthony Wood ""Archie""",male,NaN,0,0,239854,0.0000,Missing,S,1,M
527,528,first,"Farthing, Mr. John",male,NaN,0,0,PC 17483,221.7792,C95,S,1,C
855,856,third,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.3500,Missing,S,2,M
801,802,second,"Collyer, Mrs. Harvey (Charlotte Annie Tate)",female,31.0,1,1,C.A. 31921,26.2500,Missing,S,3,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,360,third,"Mockler, Miss. Helen Mary ""Ellie""",female,NaN,0,0,330980,7.8792,Missing,Q,1,M
258,259,first,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,Missing,C,1,M
736,737,third,"Ford, Mrs. Edward (Margaret Ann Watson)",female,48.0,1,3,W./C. 6608,34.3750,Missing,S,5,M
462,463,first,"Gee, Mr. Arthur H",male,47.0,0,0,111320,38.5000,E63,S,1,E


In [13]:
y_train

,Survived
692,yes
481,no
527,no
855,yes
801,yes
...,...
359,yes
258,yes
736,no
462,no


In [14]:
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = ( X_train['Age'] - mean_age ) / std_age

X_train.sample(3)

musk = (abs(X_train['Z_score']) <= 3) | (X_train['Age'].isna())

X_train = X_train[musk]

y_train = y_train[musk]

y_train.shape

(710,)

In [15]:
X_train['Fare'].describe()

,Fare
count,710.000000
mean,31.856255
std,48.118306
min,0.000000
25%,7.895800
50%,14.454200
75%,31.000000
max,512.329200


In [16]:
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

IQR = fare_Q3 - fare_Q1

minimum = max( 0, fare_Q1 - 1.5 * IQR )
maximum = fare_Q3 + 1.5 * IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


/tmp/ipykernel_641/3069097023.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)


In [17]:
X_train['Fare'].describe()

,Fare
count,710.000000
mean,23.972538
std,20.531581
min,0.000000
25%,7.895800
50%,14.454200
75%,31.000000
max,65.656300


In [18]:
# pipeline

# numerical categories

p1 = Pipeline(
    steps = [
        ( 'imputer', SimpleImputer(strategy = 'mean') ),
        ( 'scaler', StandardScaler() )
    ]
)

p1

p2 = Pipeline(
    steps = [
         ( 'imputer', SimpleImputer(strategy = 'median') ),
         ( 'scaler', MinMaxScaler() )
    ]
)

p2



Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', MinMaxScaler())])

In [19]:
# categories goes one after another via strictly following the order of the input

categories = [['third', 'second', 'first']]

In [20]:
# pipeline

# Categorical columns

p3 = Pipeline(
    steps = [
        ( 'imputer', SimpleImputer(strategy = 'most_frequent') ),
        ( 'encoder', OneHotEncoder(sparse_output = False, drop = 'first', handle_unknown = 'ignore') )
    ]
)

p3

p4 = Pipeline(
    steps = [
         ( 'imputer', SimpleImputer(strategy = 'most_frequent') ),
         ( 'encoder', OrdinalEncoder(categories = categories) ),
         ( 'scaler', MinMaxScaler() )
    ]
)

p4


Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder',
                 OrdinalEncoder(categories=[['third', 'second', 'first']])),
                ('scaler', MinMaxScaler())])

In [21]:
preprocessor = ColumnTransformer(
    transformers = [
        ( 'pipeline_1', p1, ['Age'] ),
        ( 'pipeline_2', p2, ['Fare', 'Family_size'] ),
        ( 'pipeline_3', p3, ['Embarked', 'Sex', 'Deck'] ),
        ( 'pipeline_4', p4, ['Pclass'] )
    ],
    remainder = 'drop'
)

preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

In [22]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform( y_train )
y_test = le.transform( y_test )

print(y_test)

[0 0 1 0 1 1 1 0 0 0 0 0 1 0 0 0 0 0 0 1 0 1 0 1 0 1 0 1 1 0 0 1 1 0 1 1 1
 0 0 1 0 0 0 1 1 0 0 0 1 1 1 0 0 1 1 0 0 1 1 1 0 1 0 0 1 0 1 0 0 0 1 0 0 1
 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 0 1 0 0 1 1 0 0 1 0 1 1 1 0 1 0 0 0 0 0
 0 1 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 1 0 1 0 1 0 1 1 0 0 1 0 0 1 0 0
 1 1 1 0 1 1 0 1 0 0 0 1 1 1 1 1 1 0 0 0 0 1 0 0 0 0 1 0 0 1 0]


In [23]:
lr_model = Pipeline(
    steps = [
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight = 'balanced', max_iter = 1000))
    ]
)

lr_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('pipeline_1',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age']),
                                                 ('pipeline_2',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Fare', 'Family_size']),
                                                 ('pipeline_3',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strateg...
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Embarked', 'Sex', 'Deck']),
                                                 ('pipeline_4',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OrdinalEncoder(categories=[['third',
                                                                                               'second',
                                                                                               'first']])),
                                                                  ('scaler',
                                                                   MinMaxScaler())]),
                                                  ['Pclass'])])),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [24]:
lr_model.fit( X_train, y_train )

lr_model['model'].coef_
lr_model['model'].intercept_
lr_model['model'].classes_

array([0, 1])

In [25]:
X_test

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_size,Deck
565,566,third,"Davies, Mr. Alfred J",male,24.0,2,0,A/4 48871,24.1500,Missing,S,3,M
160,161,third,"Cribb, Mr. John Hatfield",male,44.0,0,1,371362,16.1000,Missing,S,2,M
553,554,third,"Leeni, Mr. Fahim (""Philip Zenni"")",male,22.0,0,0,2620,7.2250,Missing,C,1,M
860,861,third,"Hansen, Mr. Claus Peter",male,41.0,2,0,350026,14.1083,Missing,S,3,M
241,242,third,"Murphy, Miss. Katherine ""Kate""",female,NaN,1,0,367230,15.5000,Missing,Q,2,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...
880,881,second,"Shelley, Mrs. William (Imanita Parrish Hall)",female,25.0,0,1,230433,26.0000,Missing,S,2,M
91,92,third,"Andreasson, Mr. Paul Edvin",male,20.0,0,0,347466,7.8542,Missing,S,1,M
883,884,second,"Banfield, Mr. Frederick James",male,28.0,0,0,C.A./SOTON 34068,10.5000,Missing,S,1,M
473,474,second,"Jerwan, Mrs. Amin S (Marie Marthe Thuillard)",female,23.0,0,0,SC/AH Basle 541,13.7917,D,C,1,D


In [26]:
y_pred = lr_model.predict(X_test)

lr_model.predict_proba(X_test)

array([[0.85184822, 0.14815178],
       [0.92381267, 0.07618733],
       [0.74999636, 0.25000364],
       [0.92784496, 0.07215504],
       [0.23055057, 0.76944943],
       [0.41310798, 0.58689202],
       [0.25798881, 0.74201119],
       [0.63580617, 0.36419383],
       [0.38487586, 0.61512414],
       [0.79765341, 0.20234659],
       [0.77567764, 0.22432236],
       [0.81767531, 0.18232469],
       [0.31956848, 0.68043152],
       [0.69282468, 0.30717532],
       [0.07105084, 0.92894916],
       [0.74149993, 0.25850007],
       [0.44110064, 0.55889936],
       [0.8650511 , 0.1349489 ],
       [0.79573328, 0.20426672],
       [0.2428737 , 0.7571263 ],
       [0.8650511 , 0.1349489 ],
       [0.19913988, 0.80086012],
       [0.86672074, 0.13327926],
       [0.43294304, 0.56705696],
       [0.86224898, 0.13775102],
       [0.02251829, 0.97748171],
       [0.50558952, 0.49441048],
       [0.62289687, 0.37710313],
       [0.79777558, 0.20222442],
       [0.7891374 , 0.2108626 ],
       [0.

In [27]:
accuracy = accuracy_score( y_test, y_pred )

precision = precision_score( y_test, y_pred )

recall = recall_score( y_test, y_pred )

print(f'Accuracy  : {round(accuracy, 2)}')
print(f'Precision : {round(precision, 2)}')
print(f'Recall    : {round(recall, 2)}')

Accuracy  : 0.76
Precision : 0.67
Recall    : 0.75
